# Plot Dispersion Curves Together (Multiple MAT Files)

This notebook loads multiple PINN result `.mat` files and overlays their extracted dispersion ridge curves in one figure.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

# Ensure local module import works when notebook is in Dispersion/
sys.path.append(str(Path('.').resolve()))

from pinn_dispersion_from_mat import (
    load_pinn_results,
    resample_to_uniform,
    compute_spectrum,
    extract_ridge,
    linear_dispersion,
)

In [ ]:
# ---------------- USER SETTINGS ----------------
# Add your files here (relative to this notebook in Dispersion/)
mat_files = [
    '../Result/pinn_data.mat',
    # '../Result/new_data.mat',
]

# Fallback: if only one default file is listed, auto-add all .mat files in Result/
if len(mat_files) == 1 and mat_files[0] == '../Result/pinn_data.mat':
    all_mats = sorted(Path('../Result').glob('*.mat'))
    if len(all_mats) > 1:
        mat_files = [str(p) for p in all_mats]

# Parameters for dispersion processing
k_coupling = 100.0   # update to your model
mx = 1.0             # update to your model
n_harmonic = 4
pts_per_cycle = 12
skip_transient = 0.20
omega_min = 0.01

print('MAT files to process:')
for f in mat_files:
    print('  -', f)

In [ ]:
# Process each MAT file and extract ridge
results = []
omega_max_lin = linear_dispersion(np.pi, k_coupling, mx)

for fp in mat_files:
    t_total, x_PINN_total, params = load_pinn_results(fp)

    # load_pinn_results returns (N_time, n_dof); resampler expects (n_dof, M)
    X_raw = x_PINN_total.T

    t_u, X_u, dt, omega_nyq = resample_to_uniform(
        t_total, X_raw, omega_max_lin,
        n_harmonic=n_harmonic, pts_per_cycle=pts_per_cycle
    )

    k_pos, omega_pos, spectrum = compute_spectrum(
        t_u, X_u, skip_transient=skip_transient
    )

    k_ridge, omega_ridge = extract_ridge(k_pos, omega_pos, spectrum, omega_min=omega_min)

    label = Path(fp).stem
    if 'phi2' in params:
        label += f" (phi2={params['phi2']:.3g})"

    results.append({
        'file': fp,
        'label': label,
        'k': k_ridge,
        'omega': omega_ridge,
    })

print(f'Processed {len(results)} file(s).')

In [ ]:
# Overlay plot: dispersion ridge curves together
plt.figure(figsize=(8, 5.5))

for r in results:
    valid = np.isfinite(r['omega'])
    plt.plot(r['k'][valid] / np.pi, r['omega'][valid], linewidth=2, label=r['label'])

# Optional linear reference branch
k_line = np.linspace(0, np.pi, 300)
plt.plot(k_line / np.pi, linear_dispersion(k_line, k_coupling, mx), 'k--', linewidth=1.8, label='linear reference')

plt.xlabel(r'Wavenumber $k/\pi$')
plt.ylabel(r'Frequency $\omega$ (rad/s)')
plt.title('Overlay of Dispersion Ridge Curves from Multiple MAT Files')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()